In [ ]:
# Dict comprehensions over known keys (fork dev 174a90fbd). Hover the variables.
# Keys are known when they're fixed before the loop runs: one `for`, no `if`, key = the loop variable.
splits = {'train': [1, 2, 3], 'val': [4]}

# Over a dict with known keys
sizes = {s: len(splits[s]) for s in splits}
sizes  # dict[{'train': int, 'val': int}]

In [ ]:
# .keys() and .items() work too
halves = {s: n / 2 for s, n in sizes.items()}
flags = {s: False for s in splits.keys()}
halves, flags  # dict[{'train': float, 'val': float}], dict[{'train': bool, 'val': bool}]

In [ ]:
# A list or tuple of string literals written in place
lr = {opt: 1e-3 for opt in ['adam', 'sgd']}
lr['adam']  # completion offers 'adam' and 'sgd'; lr is dict[{'adam': float, 'sgd': float}]

In [ ]:
# Pairs written in place: the first item is the key
w_trained, w_untrained = 2.0, 0.0
scaled = {name: w for name, w in [('trained', w_trained), ('untrained', w_untrained)]}
scaled  # dict[{'trained': float, 'untrained': float}]

In [ ]:
# Nested, like your training-notebook cell
z = {
    name: {s: [w * x for x in splits[s]] for s in splits}
    for name, w in [('trained', w_trained), ('untrained', w_untrained)]
}
z['trained']['val']  # hover z: both levels keep their keys

In [ ]:
# Not covered: these stay plain dict[str, ...], as before
names = ['a', 'b']  # a variable list: its contents could change
plain1 = {s: 0 for s in names}
plain2 = {s: 0 for s in splits if s != 'val'}  # an `if` clause
plain3 = {s.upper(): 0 for s in splits}  # the key isn't the loop variable

# Limit: every key gets the same value type (the value expression is checked once)
mixed = frozendict(n=1, s='x')  # frozendict[{'n': int, 's': str}]
copy = {k: mixed[k] for k in mixed}
copy  # dict[{'n': int | str, 's': int | str}], not {'n': int, 's': str}

In [ ]:
# Dicts in a list keep their keys (fork dev, uncommitted). Hover the variables.
rows = [{'loss': 0.4, 'tag': 'a'}, {'loss': 0.5, 'tag': 'b'}]
rows[0]['loss']  # float; rows is list[dict[{'loss': float, 'tag': str}]]

# A write through the list makes it plain: the checker can't tell which dict changed
rows2 = [{'loss': 0.4, 'tag': 'a'}, {'loss': 0.5, 'tag': 'b'}]
for i in range(len(rows2)):
    rows2[i]['loss'] = 'n/a'
rows2  # list[dict[str, Unknown]], plain as before

# Known hole: a write through another name (r) isn't seen, so the keys go stale
for r in rows:
    r['loss'] = 'n/a'
rows[1][
    'loss'
]  # still says float, but it's 'n/a' (same hole as passing a dict to a function that changes it)

In [ ]:
results = [{'loss': 0.4, 'tag': 'a'}, {'loss': 0.5, 'tag': 'b'}]
results.append({'loss': 0.5, 'tag': 'b'})


results

[{'loss': 0.4, 'tag': 'a'},
 {'loss': 0.5, 'tag': 'b'},
 {'loss': 0.5, 'tag': 'b'}]

In [ ]:
# Option A: a dict's own keys follow writes to its own name;
# dicts inside it (or inside a list/tuple) keep the keys they were built with.
# Comments show A. "today:" marks where the current hover differs.

# --- list of dicts ---
rows = [{'loss': 0.4, 'tag': 'a'}, {'loss': 0.5, 'tag': 'b'}]
rows  # list[dict[{'loss': float, 'tag': str}]]

rows.append({'loss': 0.6, 'tag': 'c'})
rows  # list[dict[{'loss': float, 'tag': str}]]         today: list[dict[str, Unknown]]

rows[0]['loss'] = 'n/a'
rows[0]['loss']  # Literal['n/a']: pyright narrows that exact expression, as for any write
rows[1]['loss']  # float (stale for rows[0], not tracked)  today: Unknown

for r in rows:
    r['tag'] = 1
rows  # list[dict[{'loss': float, 'tag': str}]] (not tracked)  today: plain, from the append above

# --- dict comprehension, outer keys known ---
by_run = {name: {'best': 0.4, 'epoch': 3} for name in ['a', 'b']}
by_run  # dict[{'a': dict[{'best': float, 'epoch': int}], 'b': ...}]

by_run['c'] = {'best': 0.1, 'epoch': 9}
by_run  # 'c' added: the outer dict's own keys follow writes to by_run (as today)

by_run['a']['best'] = 'n/a'
by_run  # 'a' still dict[{'best': float, ...}]            today: 'a' becomes {'best': str, ...}
by_run['a']['best']  # Literal['n/a'] (narrowing of that exact expression)

# --- dict comprehension, outer keys not known ---
def summarize_all(runs: dict[str, list[float]]):
    out = {name: {'best': min(l), 'epoch': 0} for name, l in runs.items()}
    out  # dict[str, dict[{'best': float, 'epoch': int}]]
    out['x']['best'] = 'n/a'
    out  # dict[str, dict[{'best': float, 'epoch': int}]]  today: dict[str, dict[str, Unknown]]
    return out